# Automatic Audio Description Generation  for YouTube Shorts
##  Data Collection Pipeline

**Author:** Spoorti Halappanavar  
**Student ID:** 6950243  
**Supervisor:** Dr Diptesh Kanojia  
**University:** University of Surrey  



### Overview
This notebook collects cooking YouTube Shorts
using the YouTube Data API v3. It covers:

1. **Setup** : Install libraries and configure API
2. **Data Collection** : Fetch videos from 17 channels
3. **Data Cleaning**: Filter and label cooking content

### Requirements
- YouTube Data API v3 key
- Python libraries: google-api-python-client,
  pandas, isodate

### Output
- cooking_shorts_full.csv -> Raw collected videos
- cooking_all.csv -> Filtered cooking videos
- cooking_clean.csv -> Final clean dataset

##Setup Section

In [2]:
pip -q install google-api-python-client pandas isodate

In [4]:
from google.colab import userdata
from googleapiclient.discovery import build
import pandas as pd
import isodate
import time
import os

API_KEY = userdata.get('YOUR_API_KEY_HERE')

youtube = build('youtube', 'v3', developerKey=API_KEY)
print("Youtube client ready!")

Youtube client ready!


##Data Collection Section

In [ ]:
# Cooking Channels List
COOKING_CHANNELS = [

    "Silently cooking",
    "thecookingshorts",
    "buzzfeedtasty",
    "Mob Kitchen",
    "Joshua Weissman",
    "Ethan Chlebowski",
    "Internet Shaquille",
    "Guga Foods",
    "Pro Home Cooks",
    "Binging with Babish",
    "Jamie Oliver",
    "Gordon Ramsay",
    "Sorted Food",
    "Marion's Kitchen",
    "Seonkyoung Longest",
    "Andy Cooks",
    "Nick DiGiovanni",
]

TARGET = 200

In [ ]:
def get_channel_id(channel_name):
    try:
        response = youtube.search().list(
            part="snippet",
            q=channel_name,
            type="channel",
            maxResults=1
        ).execute()
        if response["items"]:
            channel_id = response["items"][0]["id"]["channelId"]
            title = response["items"][0]["snippet"]["title"]
            print(f"   Found: {title}")
            return channel_id
        return None
    except Exception as e:
        print(f"   Error: {e}")
        return None

def get_uploads_playlist(channel_id):
    response = youtube.channels().list(
        part="contentDetails",
        id=channel_id
    ).execute()
    return response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

def get_video_ids(playlist_id, max_videos=200):
    video_ids = []
    next_page_token = None
    while len(video_ids) < max_videos:
        response = youtube.playlistItems().list(
            part="contentDetails",
            playlistId=playlist_id,
            maxResults=50,
            pageToken=next_page_token
        ).execute()
        for item in response["items"]:
            video_ids.append(item["contentDetails"]["videoId"])
        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break
    return video_ids

def get_video_details(video_ids):
    all_videos = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(batch)
        ).execute()
        for item in response["items"]:
            duration_iso = item["contentDetails"]["duration"]
            duration_sec = int(isodate.parse_duration(
                duration_iso).total_seconds())

            #   60 to 180 seconds only
            if 60 <= duration_sec <= 180:
                stats = item.get("statistics", {})
                snippet = item["snippet"]
                title = snippet.get("title", "").lower()

                # instructional content
                instructional_keywords = [
                    "recipe", "cook", "make", "how to",
                    "easy", "quick", "simple", "tutorial",
                    "step", "tips", "guide", "learn",
                    "meal", "dish", "food", "kitchen"
                ]
                is_instructional = any(
                    kw in title for kw in instructional_keywords
                )

                video_data = {
                    "video_id": item["id"],
                    "title": snippet.get("title", ""),
                    "channel_name": snippet.get("channelTitle", ""),
                    "published_at": snippet.get("publishedAt", ""),
                    "duration_secs": duration_sec,
                    "view_count": int(stats.get("viewCount", 0)),
                    "like_count": int(stats.get("likeCount", 0)),
                    "shorts_url": f"https://www.youtube.com/shorts/{item['id']}",
                    "category": "Cooking",
                    "is_instructional": is_instructional,
                    "annotation_status": "Pending",
                    "short_ad": "",
                    "detailed_ad": "",
                    "on_screen_text": "",
                }
                all_videos.append(video_data)
        time.sleep(0.1)
    return all_videos

#  MAIN COLLECTION
all_cooking = []

print(" Collecting Cooking Videos...")
print("=" * 50)

for channel_name in COOKING_CHANNELS:
    print(f"\n {channel_name}")

    channel_id = get_channel_id(channel_name)
    if not channel_id:
        continue

    playlist_id = get_uploads_playlist(channel_id)
    video_ids = get_video_ids(playlist_id, max_videos=200)
    print(f"   Found {len(video_ids)} videos")

    videos = get_video_details(video_ids)
    print(f"   {len(videos)} cooking Shorts collected!")

    all_cooking.extend(videos)
    time.sleep(1)

# Remove duplicates
seen = set()
unique = []
for v in all_cooking:
    if v["video_id"] not in seen:
        seen.add(v["video_id"])
        unique.append(v)

print(f"\n{'='*50}")
print(f" Total unique cooking videos: {len(unique)}")

🍳 Collecting Cooking Videos...

🔍 Silently cooking
  ✅ Found: Silently Cooking
  📹 Found 156 videos
  ✅ 83 cooking Shorts collected!

🔍 thecookingshorts
  ✅ Found: Thecookingshorts_
  📹 Found 200 videos
  ✅ 11 cooking Shorts collected!

🔍 buzzfeedtasty
  ✅ Found: Tasty
  📹 Found 200 videos
  ✅ 24 cooking Shorts collected!

🔍 Mob Kitchen
  ✅ Found: Mob - The weekly cooking app
  📹 Found 171 videos
  ✅ 12 cooking Shorts collected!

🔍 Joshua Weissman
  ✅ Found: Joshua Weissman
  📹 Found 200 videos
  ✅ 4 cooking Shorts collected!

🔍 Ethan Chlebowski
  ✅ Found: Ethan Chlebowski
  📹 Found 200 videos
  ✅ 16 cooking Shorts collected!

🔍 Internet Shaquille
  ✅ Found: Internet Shaquille
  📹 Found 200 videos
  ✅ 42 cooking Shorts collected!

🔍 Guga Foods
  ✅ Found: Guga Foods
  📹 Found 200 videos
  ✅ 13 cooking Shorts collected!

🔍 Pro Home Cooks
  ✅ Found: LifebyMikeG
  📹 Found 200 videos
  ✅ 71 cooking Shorts collected!

🔍 Binging with Babish
  ✅ Found: Binging with Babish
  📹 Found 200 videos


In [ ]:
import os
os.makedirs("dataset", exist_ok=True)

df = pd.DataFrame(unique)
df.to_csv("dataset/cooking_shorts_full.csv",
          index=False, encoding="utf-8-sig")

print(f" Saved {len(df)} videos!")
print(f"\n Stats:")
print(f"Instructional: {df['is_instructional'].sum()}")
print(f"All others:    {(~df['is_instructional']).sum()}")

✅ Saved 516 videos!

📊 Stats:
Instructional: 195
All others:    321


In [ ]:
from google.colab import files
files.download("dataset/cooking_shorts_full.csv")
print(" Downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded!


##Data Cleaning Section

In [ ]:
from google.colab import files
import pandas as pd

# Upload your CSV
uploaded = files.upload()

# Load it
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f" Loaded {len(df)} videos!")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 3 rows:")
df.head(3)

Saving cooking_shorts_full.csv to cooking_shorts_full (2).csv
✅ Loaded 516 videos!

Columns: ['video_id', 'title', 'channel_name', 'published_at', 'duration_secs', 'view_count', 'like_count', 'shorts_url', 'category', 'is_instructional', 'annotation_status', 'short_ad', 'detailed_ad', 'on_screen_text']

First 3 rows:


,video_id,title,channel_name,published_at,duration_secs,view_count,like_count,shorts_url,category,is_instructional,annotation_status,short_ad,detailed_ad,on_screen_text
0,wEMLag_SOC4,Not So Silently Cooking Short - Burger Heaven ...,Silently Cooking,2026-05-25T21:44:10Z,81,1092,16,https://www.youtube.com/shorts/wEMLag_SOC4,Cooking,True,Pending,NaN,NaN,NaN
1,lX5I4FobNM8,Slow down and build a burger,Silently Cooking,2026-05-21T10:03:27Z,92,336,14,https://www.youtube.com/shorts/lX5I4FobNM8,Cooking,False,Pending,NaN,NaN,NaN
2,a0Wb9B3oxgs,Not So Silently Cooking Short - What even is M...,Silently Cooking,2026-05-15T22:32:11Z,69,886,15,https://www.youtube.com/shorts/a0Wb9B3oxgs,Cooking,True,Pending,NaN,NaN,NaN


In [ ]:
# Fix video IDs that start with minus sign

def fix_video_id(vid):
    url = str(vid)
    return vid

df['video_id'] = df['shorts_url'].str.extract(
    r'youtube\.com/shorts/(.+)$'
)

print(f" Fixed video IDs!")
print(f"Sample IDs: {df['video_id'].head(3).tolist()}")

✅ Fixed video IDs!
Sample IDs: ['wEMLag_SOC4', 'lX5I4FobNM8', 'a0Wb9B3oxgs']


In [ ]:
# remove clearly not cooking content
remove_keywords = [
    # Not food related
    "star trek",
    "seth godin",
    "marketing",
    "live in the woods",
    "bad publicity",
    "vacation",
    "travel",
    "resort",
    "bloopers",
    "infomercial",
    "book",
    "cookbook announcement",
    "behind the scenes",

    # Drink only
    "negroni",
    "gin and tonic",
    "martini",
    "bloody mary",
    "sparkling water",
]

# Convert titles to lowercase for checking
def is_non_cooking(title):
    title_lower = str(title).lower()
    return any(kw in title_lower for kw in remove_keywords)

# Flag non-cooking videos
df['is_non_cooking'] = df['title'].apply(is_non_cooking)

# Show what will be removed
removed = df[df['is_non_cooking'] == True]
print(f" Videos to remove: {len(removed)}")
print("\nRemoving these:")
for title in removed['title'].tolist():
    print(f"   {title}")

🗑️ Videos to remove: 17

Removing these:
  ❌ Silently Cooking - Behind the Scenes in my Kitchen
  ❌ Silently Drinking - Negroni
  ❌ Silently Cooking - Martini
  ❌ Silently Cooking VHS - 1997 Late Night Infomercial
  ❌ Silently Cooking - Bloody Mary
  ❌ Silently Cutting - Bloopers Volume 1
  ❌ Silently Stirring - Gin and Tonic
  ❌ The Most Awkward Scene in Star Trek: TNG
  ❌ Bad Publicity
  ❌ Give Up & Live in the Woods
  ❌ Everything We Ate, Drank, and Learned at an All-Inclusive, Adults-Only Resort
  ❌ Seth Godin - This Is Marketing (tl;dr)
  ❌ Portable Soda Maker - Sparkling Water On The Go!
  ❌ AD | Who fancies some bloopers?!
  ❌ Behind the scenes with Sorted Food
  ❌ The one biggest travel mistake people make ❌ #travel #breakfast
  ❌ I'm Publishing My Cookbook! 🥳📖


In [ ]:
#  Physical food object as focus
#  Video impossible to understand without visuals

# Keywords that suggest good instructional cooking content
good_keywords = [
    "recipe", "cook", "make", "how to",
    "easy", "quick", "simple", "tutorial",
    "step", "tips", "guide", "learn",
    "meal", "dish", "food", "kitchen",
    "bake", "fry", "grill", "roast",
    "chop", "slice", "stir", "mix",
    "pasta", "soup", "curry", "burger",
    "chicken", "beef", "pork", "fish",
    "bread", "cake", "cookie", "pie",
    "rice", "noodle", "salad", "sauce",
    "breakfast", "lunch", "dinner",
    "ingredients", "seasoning",
]

def is_good_cooking(title):
    title_lower = str(title).lower()
    return any(kw in title_lower for kw in good_keywords)

df['is_good_cooking'] = df['title'].apply(is_good_cooking)

print(f" Good cooking videos: {df['is_good_cooking'].sum()}")
print(f" Unclear videos: {(~df['is_good_cooking']).sum()}")

✅ Good cooking videos: 291
⚠️ Unclear videos: 225


In [ ]:
# Remove non-cooking videos
df_clean = df[df['is_non_cooking'] == False].copy()

# Reset index
df_clean = df_clean.reset_index(drop=True)

# Add clean columns
df_clean['annotation_status'] = 'Pending'
df_clean['short_ad'] = ''
df_clean['detailed_ad'] = ''
df_clean['on_screen_text'] = ''

# Keep only needed columns
final_columns = [
    'video_id',
    'title',
    'channel_name',
    'published_at',
    'duration_secs',
    'view_count',
    'like_count',
    'shorts_url',
    'category',
    'is_instructional',
    'is_good_cooking',
    'annotation_status',
    'short_ad',
    'detailed_ad',
    'on_screen_text'
]

df_final = df_clean[final_columns]

print(f"\n{'='*50}")
print(f" FINAL DATASET SUMMARY")
print(f"{'='*50}")
print(f"Total videos: {len(df_final)}")
print(f"Instructional (TRUE): {df_final['is_instructional'].sum()}")
print(f"Good cooking content: {df_final['is_good_cooking'].sum()}")
print(f"\nBy channel:")
print(df_final['channel_name'].value_counts().to_string())


📊 FINAL DATASET SUMMARY
Total videos: 499
Instructional (TRUE): 186
Good cooking content: 281

By channel:
channel_name
Silently Cooking                75
LifebyMikeG                     71
Andy Cooks                      46
Binging with Babish             45
Sorted Food                     40
Seonkyoung Longest              38
Internet Shaquille              38
Marion Grasby                   31
Tasty                           24
Jamie Oliver                    18
Ethan Chlebowski                16
Gordon Ramsay                   16
Guga Foods                      13
Mob - The weekly cooking app    12
Thecookingshorts_               11
Joshua Weissman                  4
Nick DiGiovanni                  1


In [ ]:
import os
os.makedirs("dataset", exist_ok=True)

# All cooking videos (full dataset)
df_final.to_csv(
    "dataset/cooking_all.csv",
    index=False,
    encoding='utf-8-sig'
)
print(f" Saved {len(df_final)} videos → cooking_all.csv")

#  Only instructional ones
df_instructional = df_final[
    df_final['is_instructional'] == True
].copy()

df_instructional.to_csv(
    "dataset/cooking_instructional.csv",
    index=False,
    encoding='utf-8-sig'
)
print(f" Saved {len(df_instructional)} videos → cooking_instructional.csv")

✅ Saved 499 videos → cooking_all.csv
✅ Saved 186 videos → cooking_instructional.csv


In [ ]:
from google.colab import files

files.download("dataset/cooking_all.csv")
files.download("dataset/cooking_instructional.csv")
print(" Both files downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Both files downloaded!


In [ ]:
from google.colab import files
import pandas as pd

# Upload your file
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f" Loaded {len(df)} videos!")

Saving cooking_all.csv to cooking_all.csv
✅ Loaded 499 videos!


In [ ]:
# All keywords that suggest non-cooking content
remove_titles = [
    # Personal vlogs
    "i quit",
    "soul healing",
    "my safe place",
    "it can't hurt me",
    "remembering where",
    "morning routine",

    # Travel/non-food
    "from seed to cigar",
    "singapore's chinatown",
    "travel",
    "vacation",
    "japan 🇯🇵",

    # Celebrity/entertainment
    "niall horan",
    "tom holland",
    "memory lane",
    "mr beast",
    "monsta x",
    "ive's",
    "sneaky snacks",
    "snack silently",

    # Non-food products
    "cologne",
    "moisturiser",
    "cleaning sprays",
    "water tank",
    "rain water",
    "chicken water",
    "compost",

    # Garden/farming (not cooking)
    "planting",
    "garden",
    "grow",
    "seed",
    "cigar",

    # Drinks only
    "cocktail",
    "negroni",
    "martini",
    "gin and tonic",
    "sparkling water",

    # Non-food
    "star trek",
    "seth godin",
    "marketing",
    "bloopers",
    "infomercial",
    "opening jamie's italian",
    "second helpings",
    "trailer",
    "100th video",
    "trip down memory",
    "tennis court",
    "umbrella",
    "book",
    "publishing",
]

def should_remove(title):
    t = str(title).lower()
    return any(kw in t for kw in remove_titles)

# Apply filter
df['should_remove'] = df['title'].apply(should_remove)

# Show what will be removed
removed = df[df['should_remove'] == True]
print(f"\n🗑️ Videos to remove: {len(removed)}")
print("\nRemoving:")
for title in removed['title'].tolist():
    print(f"   {title}")


🗑️ Videos to remove: 32

Removing:
  ❌ Can Joohoney of MONSTA X Snack Silently?
  ❌ Can Shownu of MONSTA X Snack Silently?
  ❌ Can Hyungwon of MONSTA X Snack Silently? 🤫
  ❌ Can Minhyuk of MONSTA X Snack Silently? 🤫
  ❌ Sneaky Snacks With IVE's LIZ
  ❌ Sneaky Snacks With IVE's REI
  ❌ Sneaky Snacks With IVE’s An Yu-jin 🤫
  ❌ They Sent Me Fruit Punch Cologne
  ❌ They Sent Me a Root Beer Cologne
  ❌ Confusing Groceries: Cleaning Sprays
  ❌ This is my 100th video :)
  ❌ Using Milk to Filter Cocktails
  ❌ How to choose the right water tank?
  ❌ Only Planting One Thing In My Garden
  ❌ Growing Shiitake Mushrooms on Logs 🍄🪵
  ❌ How I Collect Rain Water for My Garden 🌧️
  ❌ 5 Ways To Compost Your Food Scraps
  ❌ Kimchi 100% from my garden (minus the salt🤦‍♂️)
  ❌ From Seed to Cigar
  ❌ The Ultimate Chicken Water System.
  ❌ These Mushrooms Grow in My Garden 🍄
  ❌ Second Helpings: Opening Jamie's Italian 2.0 | Trailer
  ❌ Second Helpings: Opening Jamie's Italian 2.0 #Shorts
  ❌ A trip down me

In [ ]:
# Keep only good videos
df_clean = df[df['should_remove'] == False].copy()
df_clean = df_clean.drop(columns=['should_remove'])
df_clean = df_clean.reset_index(drop=True)

print(f"\n{'='*50}")
print(f"BEFORE cleaning: {len(df)} videos")
print(f"AFTER cleaning:  {len(df_clean)} videos")
print(f"Removed:         {len(df) - len(df_clean)} videos")
print(f"{'='*50}")
print(f"\nBy channel:")
print(df_clean['channel_name'].value_counts().to_string())


BEFORE cleaning: 499 videos
AFTER cleaning:  467 videos
Removed:         32 videos

By channel:
channel_name
Silently Cooking                75
LifebyMikeG                     62
Andy Cooks                      46
Binging with Babish             45
Sorted Food                     39
Seonkyoung Longest              34
Internet Shaquille              33
Marion Grasby                   30
Tasty                           17
Ethan Chlebowski                16
Jamie Oliver                    16
Gordon Ramsay                   13
Guga Foods                      13
Mob - The weekly cooking app    12
Thecookingshorts_               11
Joshua Weissman                  4
Nick DiGiovanni                  1


In [ ]:
import os
os.makedirs("dataset", exist_ok=True)

df_clean.to_csv(
    "dataset/cooking_clean.csv",
    index=False,
    encoding='utf-8-sig'
)
print(f"✅ Saved {len(df_clean)} clean videos!")

from google.colab import files
files.download("dataset/cooking_clean.csv")
print("✅ Downloaded!")

✅ Saved 467 clean videos!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded!
